<a href="https://colab.research.google.com/github/VladBashara/ML_HSE_Course_Final_Task/blob/main/%D0%98%D1%82%D0%BE%D0%B3%D0%BE%D0%B2%D0%BE%D0%B5_%D0%B7%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install catboost scikit-learn pandas numpy scipy transformers datasets accelerate eml-parser beautifulsoup4 requests tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 7.4 MB/s eta 0:00:00


In [3]:
import os, re, io, glob, json, random, hashlib, tarfile, zipfile, urllib.request
import numpy as np, pandas as pd
from scipy.sparse import hstack
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (precision_recall_fscore_support, average_precision_score,
                             roc_auc_score, confusion_matrix)

from catboost import CatBoostClassifier
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

SEED=42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cpu


In [4]:
RAW = "raw"; os.makedirs(RAW, exist_ok=True)

SA_DIR = os.path.join(RAW, "spamassassin")
os.makedirs(SA_DIR, exist_ok=True)
SA_BASE = "https://spamassassin.apache.org/old/publiccorpus/"
SA_FILES = [
    "20030228_easy_ham.tar.bz2", "20030228_easy_ham_2.tar.bz2",
    "20030228_hard_ham.tar.bz2", "20030228_spam.tar.bz2",
    "20050311_spam_2.tar.bz2",
]
def download(url, dst):
    if not os.path.exists(dst):
        print("downloading", url)
        urllib.request.urlretrieve(url, dst)
    return dst

for fn in SA_FILES:
    dst = download(SA_BASE+fn, os.path.join(SA_DIR, fn))
    if fn.endswith(".tar.bz2"):
        with tarfile.open(dst, "r:bz2") as t:
            t.extractall(SA_DIR)
print("SpamAssassin extracted:", os.listdir(SA_DIR)[:5], "...")

ENRON_DIR = os.path.join(RAW, "enron"); os.makedirs(ENRON_DIR, exist_ok=True)
print("Enron: используем CSV-вариант ниже (см. ячейку загрузки).")

downloading https://spamassassin.apache.org/old/publiccorpus/20030228_easy_ham.tar.bz2


/tmp/ipykernel_1595/3591380307.py:22: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(SA_DIR)


downloading https://spamassassin.apache.org/old/publiccorpus/20030228_easy_ham_2.tar.bz2
downloading https://spamassassin.apache.org/old/publiccorpus/20030228_hard_ham.tar.bz2
downloading https://spamassassin.apache.org/old/publiccorpus/20030228_spam.tar.bz2
downloading https://spamassassin.apache.org/old/publiccorpus/20050311_spam_2.tar.bz2
SpamAssassin extracted: ['20030228_spam.tar.bz2', 'spam', '20030228_easy_ham.tar.bz2', 'spam_2', '20050311_spam_2.tar.bz2'] ...
Enron: используем CSV-вариант ниже (см. ячейку загрузки).


In [17]:
import re, hashlib
import pandas as pd
from email import policy
from email.parser import BytesParser
from bs4 import BeautifulSoup
from datasets import load_dataset

META_COLS = ["n_urls", "has_url", "n_digits", "n_excl", "n_upper", "len"]
BASE_COLS = ["text", "label", "source", "sender_domain"] + META_COLS

def parse_eml_bytes(raw):
    if isinstance(raw, str):
        raw = raw.encode("utf-8", "ignore")
    msg = BytesParser(policy=policy.default).parsebytes(raw)

    plain, html = [], []
    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_maintype() == "multipart":
                continue
            if part.get("Content-Disposition", "").startswith("attachment"):
                continue
            try:
                payload = part.get_content()
            except Exception:
                try:
                    payload = part.get_payload(decode=True).decode(
                        part.get_content_charset() or "utf-8", "ignore")
                except Exception:
                    continue
            if part.get_content_type() == "text/plain":
                plain.append(payload)
            elif part.get_content_type() == "text/html":
                html.append(payload)
    else:
        try:
            payload = msg.get_content()
        except Exception:
            payload = msg.get_payload(decode=True).decode("utf-8", "ignore")
        (html if msg.get_content_type() == "text/html" else plain).append(payload)

    body = "\n".join(plain) if plain else (
        BeautifulSoup("\n".join(html), "html.parser").get_text(" ") if html else "")
    subj = str(msg.get("Subject", ""))
    text = f"{subj}\n{body}".strip()

    sender = str(msg.get("From", "")).lower()
    dom = sender.split("@")[-1].strip("> ") if "@" in sender else ""
    urls = re.findall(r"https?://\S+|www\.\S+", body)
    meta = {
        "sender_domain": dom,
        "n_urls": len(urls),
        "has_url": int(len(urls) > 0),
        "n_digits": sum(c.isdigit() for c in body),
        "n_excl": body.count("!"),
        "n_upper": sum(c.isupper() for c in body),
        "len": len(body),
    }
    return text, meta

def eml_to_text_and_meta(path):
    with open(path, "rb") as f:
        return parse_eml_bytes(f.read())

SA_PARQUET = "hf://datasets/talby/spamassassin@refs/convert/parquet/unprocessed/train/*.parquet"
sa = load_dataset("parquet", data_files=SA_PARQUET)
sa_raw = pd.concat([sa[s].to_pandas() for s in sa.keys()], ignore_index=True)
print("SA columns:", sa_raw.columns.tolist())

spam_groups = {g for g in sa_raw["group"].unique() if "spam" in g.lower()}
spam_label_value = int(sa_raw.loc[sa_raw["group"].isin(spam_groups), "label"].mode().iloc[0])
print("spam_label_value:", spam_label_value, "| spam_groups:", spam_groups)

sa_rows = []
for _, r in sa_raw.iterrows():
    try:
        txt, meta = parse_eml_bytes(r["raw"])
    except Exception:
        continue
    if len(txt) == 0:
        continue
    y = 1 if int(r["label"]) == spam_label_value else 0
    sa_rows.append({"text": txt, "label": y, "source": "spamassassin", **meta})

sa_df = pd.DataFrame(sa_rows)
print("SpamAssassin:", len(sa_df),
      "| spam:", int((sa_df.label == 1).sum()),
      "| ham:", int((sa_df.label == 0).sum()))

ENRON_CANDIDATES = [
    "hf://datasets/talby/enron_spam@refs/convert/parquet/default/train/*.parquet",
    "hf://datasets/SetFit/enron_spam@refs/convert/parquet/default/train/*.parquet",
    "hf://datasets/cyberecrin/enron_spam@refs/convert/parquet/default/train/*.parquet",
]

en_df = None
for path in ENRON_CANDIDATES:
    try:
        en = load_dataset("parquet", data_files=path)
        if sum(len(en[s]) for s in en.keys()) == 0:
            continue
        en_df = pd.concat([en[s].to_pandas() for s in en.keys()], ignore_index=True)
        print("Enron loaded from:", path, "| rows:", len(en_df), "| cols:", en_df.columns.tolist())
        break
    except Exception as e:
        print("skip:", path, "->", str(e)[:120])

if en_df is None:
    print("Enron parquet не найден — обучаемся только на SpamAssassin.")
    en_df = pd.DataFrame(columns=BASE_COLS)
else:
    txt_col = next(c for c in ["text", "message", "content", "body", "raw"] if c in en_df.columns)
    lbl_col = next(c for c in ["label", "spam", "target"] if c in en_df.columns)

    en_rows = []
    for _, r in en_df.iterrows():
        val = r[txt_col]
        if isinstance(val, (bytes, bytearray)):
            txt, meta = parse_eml_bytes(bytes(val))
        else:
            txt = str(val)
            meta = {"sender_domain": "", "n_urls": 0, "has_url": 0,
                    "n_digits": 0, "n_excl": 0, "n_upper": 0, "len": len(txt)}
        if len(txt) == 0:
            continue
        en_rows.append({"text": txt, "label": int(r[lbl_col]), "source": "enron", **meta})

    en_df = pd.DataFrame(en_rows)
    if len(en_df):
        en_df = en_df[en_df["label"] == 0].copy()
    print("Enron ham:", len(en_df))

if len(en_df) == 0:
    try:
        url = "https://raw.githubusercontent.com/EmreYavuz/Spam-Detection/master/enron_ham.csv"
        tmp = pd.read_csv(url)
        tcol = next(c for c in tmp.columns if tmp[c].dtype == object)
        s = tmp[tcol].astype(str)
        en_df = pd.DataFrame({
            "text": s, "label": 0, "source": "enron", "sender_domain": "",
            "n_urls": 0, "has_url": 0, "n_digits": 0, "n_excl": 0, "n_upper": 0,
            "len": s.str.len(),
        })
        print("Enron fallback CSV rows:", len(en_df))
    except Exception as e:
        print("fallback CSV не сработал:", str(e)[:120])

for c in ["sender_domain"]:
    if c not in en_df: en_df[c] = ""
    en_df[c] = en_df[c].astype(str)
for c in META_COLS:
    if c not in en_df: en_df[c] = 0
    en_df[c] = pd.to_numeric(en_df[c], errors="coerce").fillna(0)

def norm_hash(t):
    return hashlib.md5(re.sub(r"\s+", " ", str(t).lower()).strip()
                       .encode("utf-8", "ignore")).hexdigest()

df = pd.concat([sa_df[BASE_COLS], en_df[BASE_COLS]], ignore_index=True)
df = df[df["text"].str.len() > 0].copy()
df["h"] = df["text"].map(norm_hash)
before = len(df)
df = df.drop_duplicates("h").drop(columns="h").reset_index(drop=True)
print(f"Итог после дедупа: {before} -> {len(df)} | "
      f"spam={int((df.label==1).sum())} ham={int((df.label==0).sum())}")
print(df.head(3))




SA columns: ['label', 'group', 'raw']
spam_label_value: 0 | spam_groups: {'spam', 'spam_2'}
SpamAssassin: 10749 | spam: 3795 | ham: 6954
skip: hf://datasets/talby/enron_spam@refs/convert/parquet/default/train/*.parquet -> At least one valid data file must be specified, all the data_files are invalid: {'train': []}


default/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 58.0MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Enron loaded from: hf://datasets/SetFit/enron_spam@refs/convert/parquet/default/train/*.parquet | rows: 31716 | cols: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date']
Enron ham: 15553
Итог после дедупа: 26302 -> 20851 | spam=1834 ham=19017
                                                text  label        source  \
0  RE: The Curse of India's Socialism\n\n> From: ...      0  spamassassin   
1  Re: ARRRGHHH Had GPG working, now it doesnt.\n...      0  spamassassin   
2  Re: bad focus/click behaviours\nOn Fri Sep 13 ...      0  spamassassin   

       sender_domain  n_urls  has_url  n_digits  n_excl  n_upper   len  
0         evergo.net       0        0        12       0       48  1574  
1      frommholz.org       2        1        27       0       53   944  
2  linuxworks.com.au       1        1        31       0       45  1499  


In [18]:
from sklearn.model_selection import train_test_split, StratifiedKFold

X = df["text"].values
y = df["label"].values
meta = df[META_COLS].values

X_tr, X_ho, y_tr, y_ho, m_tr, m_ho = train_test_split(
    X, y, meta, test_size=0.2, stratify=y, random_state=SEED)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(skf.split(X_tr, y_tr))
print("train:", len(X_tr), "| holdout:", len(X_ho), "| folds:", len(folds))
print("balance train:", dict(zip(*np.unique(y_tr, return_counts=True))))

train: 16680 | holdout: 4171 | folds: 5
balance train: {np.int64(0): np.int64(15213), np.int64(1): np.int64(1467)}


In [20]:
from sklearn.base import clone
from scipy.sparse import hstack

def make_vecs():
    vw = TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2, max_features=50000)
    vc = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2, max_features=50000)
    return vw, vc

C_GRID = [0.5, 1.0, 2.0, 4.0]
cv_res = {c: [] for c in C_GRID}

for tr_idx, va_idx in folds:
    vw, vc = make_vecs()
    Xtr = hstack([vw.fit_transform(X_tr[tr_idx]), vc.fit_transform(X_tr[tr_idx])]).tocsr()
    Xva = hstack([vw.transform(X_tr[va_idx]), vc.transform(X_tr[va_idx])]).tocsr()
    for C in C_GRID:
        clf = CalibratedClassifierCV(
            LogisticRegression(C=C, max_iter=2000, class_weight="balanced"),
            cv=3, method="sigmoid")
        clf.fit(Xtr, y_tr[tr_idx])
        s = clf.predict_proba(Xva)[:, 1]
        cv_res[C].append(average_precision_score(y_tr[va_idx], s))

for C in C_GRID:
    print(f"C={C}: PR-AUC {np.mean(cv_res[C]):.4f} ± {np.std(cv_res[C]):.4f}")
BEST_C = max(C_GRID, key=lambda c: np.mean(cv_res[c]))
print("BEST_C:", BEST_C)


vec_word, vec_char = make_vecs()
Xtr = hstack([vec_word.fit_transform(X_tr), vec_char.fit_transform(X_tr)]).tocsr()
Xho = hstack([vec_word.transform(X_ho), vec_char.transform(X_ho)]).tocsr()

layer1 = CalibratedClassifierCV(
    LogisticRegression(C=BEST_C, max_iter=2000, class_weight="balanced"),
    cv=3, method="sigmoid")
layer1.fit(Xtr, y_tr)
s1_ho = layer1.predict_proba(Xho)[:, 1]
print("Layer1 holdout PR-AUC:", round(average_precision_score(y_ho, s1_ho), 4))

C=0.5: PR-AUC 0.9844 ± 0.0077
C=1.0: PR-AUC 0.9882 ± 0.0061
C=2.0: PR-AUC 0.9908 ± 0.0049
C=4.0: PR-AUC 0.9925 ± 0.0041
BEST_C: 4.0
Layer1 holdout PR-AUC: 0.995


In [22]:
base_lr = LogisticRegression(C=BEST_C, max_iter=2000, class_weight="balanced")
base_lr.fit(Xtr, y_tr)

coefs = np.abs(base_lr.coef_).ravel()
top_k = min(2000, len(coefs))
top_idx = np.argsort(coefs)[-top_k:]
print("top_k:", len(top_idx))

def build2(meta_arr, s1, Xsp):
    return np.hstack([meta_arr, s1.reshape(-1, 1), Xsp[:, top_idx].toarray()])

F2_tr = build2(m_tr, layer1.predict_proba(Xtr)[:, 1], Xtr)
F2_ho = build2(m_ho, s1_ho, Xho)

cb_cv = []
for tr_idx, va_idx in folds:
    m = CatBoostClassifier(iterations=1500, depth=6, learning_rate=0.05,
                           loss_function="Logloss", eval_metric="PRAUC",
                           auto_class_weights="Balanced", verbose=False, random_seed=SEED)
    m.fit(F2_tr[tr_idx], y_tr[tr_idx],
          eval_set=(F2_tr[va_idx], y_tr[va_idx]),
          use_best_model=True, early_stopping_rounds=100, verbose=False)
    s = m.predict_proba(F2_tr[va_idx])[:, 1]
    cb_cv.append(average_precision_score(y_tr[va_idx], s))
print(f"CatBoost CV PR-AUC: {np.mean(cb_cv):.4f} ± {np.std(cb_cv):.4f}")

layer2 = CatBoostClassifier(iterations=1500, depth=6, learning_rate=0.05,
                            loss_function="Logloss", eval_metric="PRAUC",
                            auto_class_weights="Balanced", verbose=False, random_seed=SEED)
layer2.fit(F2_tr, y_tr, eval_set=(F2_ho, y_ho),
           use_best_model=True, early_stopping_rounds=100, verbose=False)
s2_ho = layer2.predict_proba(F2_ho)[:, 1]
print("Layer2 holdout PR-AUC:", round(average_precision_score(y_ho, s2_ho), 4))

top_k: 2000
CatBoost CV PR-AUC: 0.9997 ± 0.0003
Layer2 holdout PR-AUC: 0.9948


In [27]:
TAU_LOW, TAU_HIGH = 0.3, 0.7

def cascade2(texts, meta_arr, s1, s2_fn, tau_low=TAU_LOW, tau_high=TAU_HIGH):
    final = s1.copy()
    src = np.array(["layer1"]*len(texts), dtype=object)
    idx = np.where((s1 >= tau_low) & (s1 <= tau_high))[0]
    if len(idx):
        s2 = s2_fn(idx)
        final[idx] = s2; src[idx] = "layer2"
    return final, src

def s2_fn_ho(idx):
    top = Xho[idx][:, top_idx].toarray()
    return layer2.predict_proba(np.hstack([m_ho[idx], s1_ho[idx].reshape(-1,1), top]))[:,1]

final_ho, src_ho = cascade2(X_ho, m_ho, s1_ho, s2_fn_ho)
print("Cascade2 holdout PR-AUC:", round(average_precision_score(y_ho, final_ho), 4))
print("Источники:", pd.Series(src_ho).value_counts(normalize=True).round(3).to_dict())

def pick_threshold(scores, y, target_precision=0.98):
    best_t, best_r = 1.0, 0.0
    for t in np.unique(scores):
        pred = (scores >= t).astype(int)
        p, r, _, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
        if p >= target_precision and r > best_r:
            best_r, best_t = r, t
    return best_t

TAU_FINAL = pick_threshold(final_ho, y_ho, 0.98)
pred = (final_ho >= TAU_FINAL).astype(int)
p, r, f1, _ = precision_recall_fscore_support(y_ho, pred, average="binary")
print(f"Порог={TAU_FINAL:.3f}  P={p:.3f} R={r:.3f} F1={f1:.3f}")
print("Confusion:\n", confusion_matrix(y_ho, pred))

Cascade2 holdout PR-AUC: 0.9945
Источники: {'layer1': 0.993, 'layer2': 0.007}
Порог=0.888  P=0.980 R=0.946 F1=0.963
Confusion:
 [[3797    7]
 [  20  347]]


In [31]:
import os, glob, tarfile
import pandas as pd, numpy as np
from scipy.sparse import hstack
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             precision_recall_fscore_support, confusion_matrix)

from google.colab import files
up = files.upload()
ARCHIVE = list(up.keys())[0]
os.makedirs("emls", exist_ok=True)

with tarfile.open(ARCHIVE, "r:*") as t:
    t.extractall("emls")
print("top-level:", os.listdir("emls")[:10])

def find_dir(root, names):
    for n in names:
        hits = [h for h in glob.glob(f"{root}/**/{n}", recursive=True) if os.path.isdir(h)]
        if hits:
            return sorted(hits, key=len)[0]
    return None

SPAM_DIR = find_dir("emls", ["spam", "spam_2"])
HAM_DIR  = find_dir("emls", ["ham", "easy_ham", "hard_ham"])
print("SPAM_DIR:", SPAM_DIR, "| HAM_DIR:", HAM_DIR)
assert SPAM_DIR and HAM_DIR, "Не найдены папки spam/ и ham/ в архиве — проверьте структуру"

def parse_eml_flat(path, label):
    txt, meta = eml_to_text_and_meta(path)
    return {"text": txt, "label": label, "path": path, **meta}

rows = []
for d, lab in [(SPAM_DIR, 1), (HAM_DIR, 0)]:
    for p in glob.glob(f"{d}/**/*.eml", recursive=True):
        try:
            r = parse_eml_flat(p, lab)
            if len(r["text"]) > 0:
                rows.append(r)
        except Exception:
            pass

eml_df = pd.DataFrame(rows).reset_index(drop=True)
print("eml test:", len(eml_df),
      "| spam:", int((eml_df.label==1).sum()),
      "| ham:", int((eml_df.label==0).sum()))

for c in META_COLS:
    if c not in eml_df: eml_df[c] = 0
    eml_df[c] = pd.to_numeric(eml_df[c], errors="coerce").fillna(0)

Xe = eml_df["text"].values
ye = eml_df["label"].values
me = eml_df[META_COLS].values

Xe_sp = hstack([vec_word.transform(Xe), vec_char.transform(Xe)]).tocsr()
s1_e = layer1.predict_proba(Xe_sp)[:, 1]

def s2_fn_e(idx):
    top = Xe_sp[idx][:, top_idx].toarray()
    return layer2.predict_proba(np.hstack([me[idx], s1_e[idx].reshape(-1,1), top]))[:, 1]

final_e, src_e = cascade2(Xe, me, s1_e, s2_fn_e)
print("EML PR-AUC:", round(average_precision_score(ye, final_e), 4),
      "| ROC-AUC:", round(roc_auc_score(ye, final_e), 4))

pred = (final_e >= TAU_FINAL).astype(int)
p, r, f1, _ = precision_recall_fscore_support(ye, pred, average="binary")
print(f"Порог={TAU_FINAL:.3f}  P={p:.3f} R={r:.3f} F1={f1:.3f}")
print("Confusion:\n", confusion_matrix(ye, pred))
print("Источники:", pd.Series(src_e).value_counts(normalize=True).round(3).to_dict())

Saving test_task_emails.tar.xz to test_task_emails.tar (1).xz


/tmp/ipykernel_1595/1163111905.py:16: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall("emls")


top-level: ['spam', 'ham']
SPAM_DIR: emls/spam | HAM_DIR: emls/ham
eml test: 1598 | spam: 998 | ham: 600
EML PR-AUC: 0.9945 | ROC-AUC: 0.9906
Порог=0.888  P=0.991 R=0.797 F1=0.883
Confusion:
 [[593   7]
 [203 795]]
Источники: {'layer1': 0.927, 'layer2': 0.073}


In [32]:
err = eml_df.copy()
err["score"] = final_e
err["source"] = src_e
err["pred"] = (err["score"] >= TAU_FINAL).astype(int)

print("FP (ham→spam):")
print(err[(err.label==0) & (err.pred==1)]
      .sort_values("score", ascending=False)
      [["path","score","source","sender_domain"]].head(10).to_string())

print("\nFN (spam→ham):")
print(err[(err.label==1) & (err.pred==0)]
      .sort_values("score")
      [["path","score","source","sender_domain"]].head(10).to_string())

print("\nПо источникам:")
print(err.groupby(["source","label"]).size().unstack(fill_value=0))

FP (ham→spam):
                  path     score  source         sender_domain
1211  emls/ham/328.eml  0.957514  layer2          id.apple.com
1369  emls/ham/292.eml  0.951759  layer2         jetbrains.com
1457  emls/ham/233.eml  0.947855  layer2          roboflow.com
1299  emls/ham/475.eml  0.946295  layer2    pythonanywhere.com
1061  emls/ham/578.eml  0.943174  layer2            kaggle.com
1075  emls/ham/220.eml  0.938169  layer2         sololearn.com
1442  emls/ham/540.eml  0.930722  layer2  en.sbermegamarket.ru

FN (spam→ham):
                  path     score  source     sender_domain
45   emls/spam/317.eml  0.000604  layer1  fintech-start.ru
555  emls/spam/795.eml  0.001709  layer1            53.com
405  emls/spam/303.eml  0.001712  layer1           bbt.com
62   emls/spam/337.eml  0.001883  layer1           bbt.com
874  emls/spam/604.eml  0.002340  layer1            53.com
565  emls/spam/477.eml  0.003136  layer1   security.53.com
11   emls/spam/809.eml  0.004297  layer1          eb

In [33]:
import os, joblib
ART = "artifacts"; os.makedirs(ART, exist_ok=True)

joblib.dump(layer1, f"{ART}/layer1_logreg.joblib")
joblib.dump((vec_word, vec_char), f"{ART}/tfidf.joblib")
joblib.dump({"top_idx": top_idx, "META_COLS": META_COLS,
             "TAU_LOW": TAU_LOW, "TAU_HIGH": TAU_HIGH,
             "TAU_FINAL": float(TAU_FINAL)}, f"{ART}/config.joblib")
layer2.save_model(f"{ART}/layer2_catboost.cbm")
print("saved:", os.listdir(ART))

saved: ['layer1_logreg.joblib', 'layer2_catboost.cbm', 'tfidf.joblib', 'config.joblib']


In [34]:
import numpy as np, joblib
from scipy.sparse import hstack
from catboost import CatBoostClassifier

L1 = joblib.load(f"{ART}/layer1_logreg.joblib")
VW, VC = joblib.load(f"{ART}/tfidf.joblib")
CFG = joblib.load(f"{ART}/config.joblib")
L2 = CatBoostClassifier(); L2.load_model(f"{ART}/layer2_catboost.cbm")

sample_idx = np.random.RandomState(0).choice(len(Xe), size=min(5, len(Xe)), replace=False)
Xs = Xe[sample_idx]; Ms = me[sample_idx]

Xsp_live = hstack([vec_word.transform(Xs), vec_char.transform(Xs)]).tocsr()
s1_live = layer1.predict_proba(Xsp_live)[:,1]
top_live = Xsp_live[:, top_idx].toarray()
s2_live = layer2.predict_proba(np.hstack([Ms, s1_live.reshape(-1,1), top_live]))[:,1]

Xsp_saved = hstack([VW.transform(Xs), VC.transform(Xs)]).tocsr()
s1_saved = L1.predict_proba(Xsp_saved)[:,1]
top_saved = Xsp_saved[:, CFG["top_idx"]].toarray()
s2_saved = L2.predict_proba(np.hstack([Ms, s1_saved.reshape(-1,1), top_saved]))[:,1]

print("layer1 max|diff|:", float(np.max(np.abs(s1_live - s1_saved))))
print("layer2 max|diff|:", float(np.max(np.abs(s2_live - s2_saved))))
assert np.allclose(s1_live, s1_saved, atol=1e-6), "layer1 расходится!"
assert np.allclose(s2_live, s2_saved, atol=1e-6), "layer2 расходится!"
print("OK: артефакты воспроизводят инференс 1-в-1")

layer1 max|diff|: 0.0
layer2 max|diff|: 0.0
OK: артефакты воспроизводят инференс 1-в-1


In [37]:
import os, textwrap
SVC = "spam-service"
os.makedirs(f"{SVC}/app", exist_ok=True)

with open(f"{SVC}/app/eml_parser.py","w") as f:
    f.write(textwrap.dedent('''
    import re
    from email import policy
    from email.parser import BytesParser
    from bs4 import BeautifulSoup

    def parse_eml_bytes(raw):
        if isinstance(raw,str): raw = raw.encode("utf-8","ignore")
        msg = BytesParser(policy=policy.default).parsebytes(raw)
        plain, html = [], []
        if msg.is_multipart():
            for part in msg.walk():
                if part.get_content_maintype()=="multipart": continue
                if part.get("Content-Disposition","").startswith("attachment"): continue
                try: payload = part.get_content()
                except Exception:
                    try: payload = part.get_payload(decode=True).decode(part.get_content_charset() or "utf-8","ignore")
                    except Exception: continue
                if part.get_content_type()=="text/plain": plain.append(payload)
                elif part.get_content_type()=="text/html": html.append(payload)
        else:
            try: payload = msg.get_content()
            except Exception: payload = msg.get_payload(decode=True).decode("utf-8","ignore")
            (html if msg.get_content_type()=="text/html" else plain).append(payload)
        body = "\\n".join(plain) if plain else (BeautifulSoup("\\n".join(html),"html.parser").get_text(" ") if html else "")
        subj = str(msg.get("Subject",""))
        text = f"{subj}\\n{body}".strip()
        sender = str(msg.get("From","")).lower()
        dom = sender.split("@")[-1].strip("> ") if "@" in sender else ""
        urls = re.findall(r"https?://\\S+|www\\.\\S+", body)
        meta = {"sender_domain":dom,"n_urls":len(urls),"has_url":int(len(urls)>0),
                "n_digits":sum(c.isdigit() for c in body),"n_excl":body.count("!"),
                "n_upper":sum(c.isupper() for c in body),"len":len(body)}
        return text, meta
    ''').lstrip())

with open(f"{SVC}/app/model.py","w") as f:
    f.write(textwrap.dedent('''
    import os, joblib, numpy as np
    from scipy.sparse import hstack
    from catboost import CatBoostClassifier

    ART = os.getenv("ARTIFACTS_DIR","artifacts")

    class SpamCascade:
        def __init__(self):
            self.layer1 = joblib.load(f"{ART}/layer1_logreg.joblib")
            self.vec_word, self.vec_char = joblib.load(f"{ART}/tfidf.joblib")
            cfg = joblib.load(f"{ART}/config.joblib")
            self.top_idx=cfg["top_idx"]; self.META_COLS=cfg["META_COLS"]
            self.TAU_LOW=cfg["TAU_LOW"]; self.TAU_HIGH=cfg["TAU_HIGH"]; self.TAU_FINAL=cfg["TAU_FINAL"]
            self.layer2 = CatBoostClassifier(); self.layer2.load_model(f"{ART}/layer2_catboost.cbm")

        def predict(self, texts, metas):
            meta = np.array([[m[c] for c in self.META_COLS] for m in metas], dtype=float)
            X = hstack([self.vec_word.transform(texts), self.vec_char.transform(texts)]).tocsr()
            s1 = self.layer1.predict_proba(X)[:,1]
            final = s1.copy(); source = np.array(["layer1"]*len(texts), dtype=object)
            idx = np.where((s1>=self.TAU_LOW)&(s1<=self.TAU_HIGH))[0]
            if len(idx):
                top = X[idx][:, self.top_idx].toarray()
                s2 = self.layer2.predict_proba(np.hstack([meta[idx], s1[idx].reshape(-1,1), top]))[:,1]
                final[idx]=s2; source[idx]="layer2"
            label = (final>=self.TAU_FINAL).astype(int)
            return final, label, source
    ''').lstrip())

with open(f"{SVC}/app/main.py","w") as f:
    f.write(textwrap.dedent('''
    from fastapi import FastAPI, UploadFile, File, HTTPException
    from pydantic import BaseModel
    from app.model import SpamCascade
    from app.eml_parser import parse_eml_bytes

    app = FastAPI(title="Spam Cascade API (2-layer)", version="1.0")
    clf = SpamCascade()

    class TextIn(BaseModel):
        text: str
        sender_domain: str = ""

    @app.get("/health")
    def health(): return {"status":"ok"}

    @app.post("/predict")
    def predict(p: TextIn):
        meta = {"sender_domain":p.sender_domain,"n_urls":0,"has_url":0,
                "n_digits":sum(c.isdigit() for c in p.text),
                "n_excl":p.text.count("!"),"n_upper":sum(c.isupper() for c in p.text),
                "len":len(p.text)}
        s,l,src = clf.predict([p.text],[meta])
        return {"score":float(s[0]),"label":int(l[0]),"is_spam":bool(l[0]),"source":src[0]}

    @app.post("/predict_eml")
    async def predict_eml(file: UploadFile = File(...)):
        if not file.filename.lower().endswith(".eml"): raise HTTPException(400,"need .eml")
        text, meta = parse_eml_bytes(await file.read())
        if not text: raise HTTPException(400,"empty email")
        s,l,src = clf.predict([text],[meta])
        return {"score":float(s[0]),"label":int(l[0]),"is_spam":bool(l[0]),"source":src[0],"sender_domain":meta["sender_domain"]}
    ''').lstrip())

with open(f"{SVC}/requirements.txt","w") as f:
    f.write("""fastapi==0.115.0
uvicorn[standard]==0.30.6
pydantic==2.9.2
numpy==1.26.4
scipy==1.13.1
scikit-learn==1.5.2
joblib==1.4.2
catboost==1.2.7
beautifulsoup4==4.12.3
python-multipart==0.0.9
""")

with open(f"{SVC}/Dockerfile","w") as f:
    f.write("""FROM python:3.11-slim
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1 ARTIFACTS_DIR=/app/artifacts
WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential libgomp1 && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app ./app
COPY artifacts ./artifacts

EXPOSE 8080
CMD ["sh","-c","uvicorn app.main:app --host 0.0.0.0 --port ${PORT:-8080}"]
""")

with open(f"{SVC}/__init__.py","w") as f:
  pass

print("service files written to", SVC)
!ls -R {SVC} | head -40

service files written to spam-service
spam-service:
app
artifacts
Dockerfile
__init__.py
requirements.txt

spam-service/app:
eml_parser.py
main.py
model.py

spam-service/artifacts:
config.joblib
layer1_logreg.joblib
layer2_catboost.cbm
tfidf.joblib


In [38]:
import shutil, os
shutil.copytree("artifacts", f"{SVC}/artifacts", dirs_exist_ok=True)
!zip -r spam-service.zip {SVC} > /dev/null
from google.colab import files
files.download("spam-service.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>